In [1]:

%pip -q install -U openai


Groq connection test

In [2]:

from getpass import getpass
from openai import OpenAI

client = OpenAI(
    api_key=getpass(""),
    base_url="https://api.groq.com/openai/v1",
    timeout=60.0,
    max_retries=2,
)

MODEL = "openai/gpt-oss-20b"

test = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "Reply with: CONNECTION OK"}
    ],
    max_tokens=50,
)

print(test.choices[0].message.content)


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··········
CONNECTION OK


Build all 6 agents

In [3]:

import re
from datetime import date

AGENTS = [
    {
        "name": "1. Patient Intake Agent",
        "instructions": """
You organize patient-provided information.
Summarize reported symptoms, duration, relevant history and unknowns.
Separate reported facts from assumptions.
Do not diagnose or invent patient details.
List questions that a clinician may need to ask.
"""
    },
    {
        "name": "2. Safety Screening Agent",
        "instructions": """
Review the provided description for possible warning signs.
Do not claim to diagnose, medically clear, or rule out an emergency.
If potentially life-threatening symptoms are described, advise immediate
local emergency assistance instead of waiting for this AI report.
State uncertainty and explain that this is not a validated triage tool.
Never use absence of listed symptoms as proof of safety.
"""
    },
    {
        "name": "3. Medication Reconciliation Agent",
        "instructions": """
Organize only the medications and allergies actually reported.
List names, doses and frequency only if supplied by the user.
Flag missing details and possible discrepancies for pharmacist or clinician
review. Do not recommend starting, stopping, or changing medicines.
Do not invent interactions or claim a medication list is complete.
"""
    },
    {
        "name": "4. Lab Report Explanation Agent",
        "instructions": """
Explain any laboratory values explicitly provided by the user in general,
educational language. Preserve the exact values and units.
Use reference ranges only if supplied. If no lab values are provided,
say that no lab results were supplied and request the actual values.
Do not diagnose or invent results.
"""
    },
    {
        "name": "5. Care Preparation Agent",
        "instructions": """
Prepare a practical checklist for a conversation with a licensed clinician.
Suggest relevant questions and information to bring, based on the supplied
case. Do not prescribe treatment or claim that an appointment is booked.
Distinguish general preparation from medical advice.
"""
    },
    {
        "name": "6. Clinical Handoff Agent",
        "instructions": """
Create a concise structured handoff for a healthcare professional using
the original case and all five previous outputs.
Include reported concern, relevant facts, unknowns, medication information
if supplied, lab information if supplied, safety concerns and questions.
Flag conflicting or unsupported claims. Do not invent missing information.
End with a clear statement that clinician review is required.
"""
    }
]


def call_agent(agent, patient_case, previous_outputs=None):
    previous_outputs = previous_outputs or {}

    context = ""
    if previous_outputs:
        context = "\n\nPrevious agent outputs (unverified):\n"
        for name, output in previous_outputs.items():
            context += f"\n--- {name} ---\n{output}\n"

    prompt = f"""
Patient case:
{patient_case}

{context}

Follow your assigned role. Treat all previous agent outputs as
unverified, and do not repeat unsupported claims.
"""

    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {
                    "role": "system",
                    "content": agent["instructions"]
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            max_tokens=1200,
        )

        output = response.choices[0].message.content
        if not output or not output.strip():
            raise RuntimeError("The model returned an empty response.")

        print(f"\n{'=' * 12} {agent['name']} {'=' * 12}")
        print(output)

        return output

    except Exception as e:
        raise RuntimeError(
            f"{agent['name']} failed: {type(e).__name__}: {e}"
        ) from e


def run_healthcare_system(patient_case):
    if not isinstance(patient_case, str) or not patient_case.strip():
        raise ValueError("Patient case cannot be empty.")

    if len(patient_case) > 8000:
        raise ValueError("For this demo, keep the case under 8000 characters.")

    outputs = {}

    # Agents 1–5 perform their individual tasks.
    for agent in AGENTS[:5]:
        outputs[agent["name"]] = call_agent(
            agent,
            patient_case,
            outputs
        )

    # Agent 6 combines the results into a handoff.
    outputs[AGENTS[5]["name"]] = call_agent(
        AGENTS[5],
        patient_case,
        outputs
    )

    report = f"""# Healthcare Coordination Report

Date: {date.today().isoformat()}

**Educational demonstration — not a diagnosis or treatment plan.**

## Original patient information
{patient_case}

""" + "\n\n".join(
        f"## {name}\n\n{output}"
        for name, output in outputs.items()
    ) + """

## Important limitations
This report is AI-generated and may be incorrect or incomplete.
It is not a substitute for assessment by a licensed healthcare professional.
No appointment, clinical record, or external medical database was accessed.
"""

    return report, outputs


print("Healthcare multi-agent system ready.")
print("Number of specialized agents:", len(AGENTS))


Healthcare multi-agent system ready.
Number of specialized agents: 6


Run the six-agent system

In [4]:

patient_case = """
Fictional educational case. No real patient data.

An adult reports a mild headache since this morning and says they
have not had much water today. Their age, medical history, current
medications, allergies, vital signs and other symptoms are unknown.

No laboratory results have been provided.

The person wants help preparing information for a healthcare visit.
"""

try:
    final_report, agent_outputs = run_healthcare_system(patient_case)
    print("\nSUCCESS: All six agents completed.")
except Exception as e:
    print("\nSYSTEM STOPPED:", str(e))
    raise



============ 1. Patient Intake Agent ============
**Patient‑Provided Information (Verified Facts)**  
- **Chief complaint:** Mild headache.  
- **Onset:** Since this morning (exact duration unknown).  
- **Self‑reported contributing factor:** Not drinking much water today.  

**Unknown / Unspecified Information**  
- Age (pediatric, adult, geriatric).  
- Sex/gender.  
- Medical history (chronic conditions, previous headaches, neurologic or ophthalmologic issues, cardiovascular disease, hypertension, etc.).  
- Current medications (prescription, OTC, supplements, herbal products).  
- Known allergies (drug, food, environmental).  
- Vital signs (blood pressure, heart rate, temperature, respiratory rate, oxygen saturation).  
- Other symptoms (nausea, vomiting, visual disturbances, photophobia, phonophobia, dizziness, neck stiffness, recent trauma, fever, rash, chest pain, palpitations).  
- Recent events or exposures (travel, infection, head injury, stressors, sleep deprivation, caffe

Download the final report

In [5]:

from google.colab import files

with open(
    "healthcare_multi_agent_report.md",
    "w",
    encoding="utf-8"
) as f:
    f.write(final_report)

files.download("healthcare_multi_agent_report.md")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>